## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到所在的代码树根 (`solutions/` 或 `tutorials/`)，
   这样 `from attention.mha import ...` 这种导入能直接生效。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd to the tree root (whichever of `solutions/` or `tutorials/` this notebook lives in), turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys

# Walk up from the notebook's CWD until we find a directory named
# `solutions` or `tutorials`. Works no matter which tree the student
# opened. 不论 notebook 位于 solutions/ 还是 tutorials/ 都能正确定位。
ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Fallback: maybe we were started at the repo root.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')

assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the confidence chapter's reference .pt files live
control_folder = 'confidence/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 5 章 · Confidence

## AF3 输出的不只是坐标

AF2 / AF3 的最大产品差异之一: **每个预测都附带置信度**。给 100 个结构告诉用户「这 100 个里哪些可信哪些瞎猜」，是这类模型在湿实验里能用起来的前提。

AF3 置信度系统输出 4 类:

| 量 | 全名 | 维度 | 含义 |
|---|---|---|---|
| **pLDDT** | predicted local-distance-difference test | per atom | 局部精度 (0-100，越高越好) |
| **PAE** | predicted aligned error | per token pair | 给定一对 token，二者相对位姿误差的分布 |
| **PDE** | predicted distance error | per token pair | 二者距离的误差分布 (对称量) |
| **resolved** | 是否在实验中可见 | per atom | 二分类: 这个原子在最终结构里有没有坐标 |

再加上**distogram** —— 训练目标之一，预测每对 token 之间的距离分布 (64 个 bin)。Distogram 不直接是置信度但在推理时也由置信度路径产出。

### 这四个量的数学定义

**LDDT** ([Mariani et al. 2013](https://academic.oup.com/bioinformatics/article/29/21/2722/195896)): 对原子 $\ell$，定义它与一组距离接触原子的相对距离误差:

$$\mathrm{LDDT}(\ell) = \frac{1}{|\mathcal{N}(\ell)|} \sum_{m \in \mathcal{N}(\ell)} \frac{1}{4} \sum_{t \in \{0.5, 1, 2, 4\}} \mathbb{1}\big[|d_{\ell m}^\text{pred} - d_{\ell m}^\text{gt}| < t\big]$$

其中 $\mathcal{N}(\ell)$ 是真实结构里离 $\ell$ 在 15Å 内的原子。本质: 把「预测距离与真实距离误差是否在 0.5/1/2/4Å 之内」做 4 个阈值平均。LDDT 范围 0-1，乘 100 报告。

AF3 不能直接算 LDDT (因为推理时不知道 ground truth)，所以预测一个**分布**:

$$\mathrm{pLDDT}_\ell^h = \mathrm{Linear}_{b\_\text{plddt}}\big(\mathrm{LN}(s_\ell)\big) \to \text{50 bins over } [0, 100]$$

(softmax 后做 bin 中心加权得到标量预测)。

**PAE (predicted aligned error)**: 对每对 token (i, j) 预测「如果把预测结构对齐到 token i 的局部坐标系，token j 的位置误差是多少」。形式化:

$$\mathrm{AE}(i, j) = \| T_i^\text{pred} \cdot x_j^\text{pred} - T_i^\text{gt} \cdot x_j^\text{gt} \|$$

其中 $T_i$ 是 token i 的局部 frame (用前面 `expressCoordinatesInFrame` 算的)。PAE 是 N×N 矩阵 (有方向)，可分到 64 个 bin。**iPTM** (interface predicted TM-score)由 PAE 在 chain 边界附近的统计算出，是判断复合物界面置信的关键。

**PDE (predicted distance error)**: 对每对 token 预测「距离误差的分布」:

$$\mathrm{DE}(i, j) = \big| \|x_i^\text{pred} - x_j^\text{pred}\| - \|x_i^\text{gt} - x_j^\text{gt}\| \big|$$

**对称**: $\mathrm{DE}(i, j) = \mathrm{DE}(j, i)$。所以 PDE head 内部要先做 `z + z.transpose(-2, -3)` 对称化再 LN + Linear。

**resolved**: 训练时把 PDB 里有坐标的原子标 1、没有的标 0 (例如 missing residue)。这是 per-atom 二分类，告诉用户「模型预测的这个原子能否在湿实验里被观察到」。

**Distogram (Alg 1 line 17)**: 训练目标。对每对 token (i, j) 直接预测真实距离落在 64 个 bin 的哪一个。由 pair 张量 z 直接投出 logits，对称化保证 $\mathrm{Dist}(i, j) = \mathrm{Dist}(j, i)$。

## 本章范围

完整的 ConfidenceHead 是个比较重的复合模块: 内部跑一个小型 PairformerStack +四个分类头。完整 forward 需要带 atom 级索引映射的特征字典，单元测试太繁琐 ——所以我们:

1. **5.1** 测最简单的入口 `DistogramHead.forward` —— 它就是一个 Linear + 对称化。
2. **5.2** 用 shape 检查覆盖整个 `ConfidenceHead.__init__` —— 验证你的   `__init__` 把所有子模块都装对了 (这是最容易出错的部分)。

ConfidenceHead 的完整 `forward` 与 `memory_efficient_forward` 在 `overview.ipynb` 里端到端验证。

## 本章模块

| 文件 | 类 | 作用 |
|---|---|---|
| `confidence/distogram_head.py` | `DistogramHead.forward` | distogram logits (Alg 1 line 17) |
| `confidence/confidence_head.py` | `ConfidenceHead.__init__` | 装配 4 个 head |

## 5.1 DistogramHead (Algorithm 1 line 17)

AF3 训练目标之一: **预测每对 token 之间真实距离落在哪个 bin**。Distogram 模型给出一个$[N, N, B]$ 张量 (B 个距离 bin 的 logits)，softmax 后得到每对 token 距离的概率分布。

DistogramHead 极简: 就是把 pair 张量过一个 Linear 投到 64 个 bin。但有两个关键点:

### 1. 零初始化

`self.linear = Linear(c_z, no_bins, initializer="zeros")` —— 训练初期输出是 0，softmax 后是均匀分布，logits 完全由训练学到。如果用默认初始化，初始预测就有偏，训练动态不稳。

### 2. 对称化

距离矩阵在数学上对称: $d(i, j) = d(j, i)$。但 pair 张量在 Pairformer 里**并不严格对称**。DistogramHead 通过 `logits = logits + logits.transpose(-2, -3)` 显式强制对称化 ——保证 distogram 预测合法。

**任务**: 打开 `confidence/distogram_head.py` 填 forward 的 TODO 块 (3 步: linear → 对称化 → 返回)。

In [ ]:
from confidence.distogram_head import DistogramHead
from confidence.control_values.confidence_checks import (
    c_z, no_bins, test_inputs,
    test_module_shape, test_module_forward,
)

dh = DistogramHead(c_z=c_z, no_bins=no_bins)
test_module_shape(dh, 'distogram_head', control_folder)
test_module_forward(
    dh, 'distogram_head',
    inputs=(test_inputs['z'],),
    output_names='out',
    control_folder=control_folder,
)
print('DistogramHead ✓')

## 5.2 ConfidenceHead 装配检查

完整的 ConfidenceHead 行为相当复杂:

```text
  x_pred_rep_coords  (per-token 代表原子 coord)
      │
      ├── cdist → 距离矩阵
      │     │
      │     └── one_hot 分箱 (distance_bin_*)
      │           │
      │           └── linear_no_bias_d / _wo_onehot → 加到 z_pair
      ▼
  z_pair + s_trunk
      │
      └── PairformerStack(c_s=c_s, c_z=c_z, n_blocks=n_blocks)
      │
      ├── pae_ln + linear_no_bias_pae  →  PAE logits  (per pair, b_pae bins)
      ├── pde_ln + linear_no_bias_pde  →  PDE logits  (per pair, b_pde bins)
      ├── plddt_ln + plddt_weight     →  pLDDT logits (per atom, b_plddt bins)
      └── resolved_ln + resolved_weight → resolved (per atom, 2 classes)
```

`__init__` 里要拼 11 个左右子模块 + 2 个特殊的 weight 张量 (`plddt_weight` 和`resolved_weight` 是 atom-slot 级权重，shape `[max_atoms_per_token, c_s, b_*]`)。**任何子模块漏建 / 命名错 / 维度错都会让 state_dict 对不上 Protenix 权重**。

因此我们用 `test_module_shape` 做整体形状检查 —— 它枚举所有命名参数的 shape，与保存的参考字典对比，任何不一致都会立刻报。

完整 `forward` 与 `memory_efficient_forward` 还需要原子级索引映射 (`atom_to_token_idx`等)，构造测试 dict 太繁琐 —— 留给端到端 `overview.ipynb`。

In [ ]:
from confidence.confidence_head import ConfidenceHead
from confidence.control_values.confidence_checks import test_module_shape

ch = ConfidenceHead(
    n_blocks=1,
    c_s=32, c_z=c_z,
    c_s_inputs=32,
    b_pae=8, b_pde=8, b_plddt=10, b_resolved=2,
    max_atoms_per_token=5,
    pairformer_dropout=0.0,
    distance_bin_start=3.25, distance_bin_end=8.25, distance_bin_step=1.25,
)
test_module_shape(ch, 'confidence_head_init', control_folder)
print('ConfidenceHead.__init__ ✓')

## 章节小结

本章你交付了:

1. **`DistogramHead.forward`** —— pair → 距离 bin logits，零初始化 + 对称化。
2. **`ConfidenceHead.__init__`** —— 4 个置信度 head + 距离 bin 投影 + 内部 PairformerStack   的完整装配，通过 state_dict shape 检查验证。

完整 `ConfidenceHead.forward` 与 `memory_efficient_forward` 的实现 TODO 已在 `confidence_head.py` 写好详细伪代码，但在端到端 `overview.ipynb` 里才整体测试。

**全部章节走完，下一站**: `overview.ipynb` —— 把所有零件装成完整 Protenix，加载字节官方权重，跑一次 7r6r 蛋白的端到端推理。pLDDT≈33 / pTM≈0.21 是tiny 模型在 CPU 上 5 步采样的预期结果 (论文 base 模型当然更高，但跑一次 30 分钟，学习用足够了)。